Cell 1 — cấu hình

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

import duckdb
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (
            (candidate / "dbt_project" / "dbt_project.yml").is_file()
            and (candidate / "data" / "raw").is_dir()
        ):
            return candidate

    raise FileNotFoundError("Không tìm thấy repository CommerceFlow360.")

PROJECT_ROOT = find_project_root()
DBT_DIR = PROJECT_ROOT / "dbt_project"
DUCKDB_PATH = Path(
    os.environ.get("DUCKDB_PATH", PROJECT_ROOT / "commerceflow.duckdb")
).resolve()

if not DUCKDB_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy DuckDB: {DUCKDB_PATH}. "
        "Hãy chạy Bronze, Silver và Gold trước."
    )

os.environ["DUCKDB_PATH"] = str(DUCKDB_PATH)

PROFILES_DIR = Path.home() / ".dbt"
PROFILES_DIR.mkdir(parents=True, exist_ok=True)

profile_source = DBT_DIR / "profiles.yml.example"
profile_target = PROFILES_DIR / "profiles.yml"

if not profile_target.exists():
    shutil.copy(profile_source, profile_target)

os.environ["DBT_PROFILES_DIR"] = str(PROFILES_DIR)

print("Project root:", PROJECT_ROOT)
print("DuckDB:", DUCKDB_PATH)

Project root: D:\KHOI\DAI HOC\TREN TRUONG\NAM3\KI 1\DATA WAREHOUSE\DOAN\CommerceFlow360
DuckDB: D:\KHOI\DAI HOC\TREN TRUONG\NAM3\KI 1\DATA WAREHOUSE\DOAN\CommerceFlow360\commerceflow.duckdb


Cell 2 — chạy test Gold

In [2]:
def run_dbt(*args):
    command = ["dbt", *args, "--no-use-colors"]

    dbt_env = os.environ.copy()
    dbt_env["PYTHONUTF8"] = "1"
    dbt_env["PYTHONIOENCODING"] = "utf-8"

    result = subprocess.run(
        command,
        cwd=DBT_DIR,
        env=dbt_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    return result


test_names = [
    "test_gold_fact_grain",
    "test_gold_dimension_relationships",
    "test_sales_reconciliation",
]

test_result = run_dbt("test", "--select", *test_names)

run_results_path = DBT_DIR / "target" / "run_results.json"
if not run_results_path.is_file():
    raise FileNotFoundError(
        f"dbt không tạo run_results.json: {run_results_path}"
    )

run_results = json.loads(run_results_path.read_text(encoding="utf-8"))

test_summary = pd.DataFrame([
    {
        "test_name": result["unique_id"].split(".")[-1],
        "status": result["status"].upper(),
        "failures": result.get("failures") or 0,
    }
    for result in run_results.get("results", [])
])

display(test_summary)

completed_tests = set(test_summary["test_name"])
missing_tests = set(test_names) - completed_tests

assert not missing_tests, f"Chưa chạy các test: {sorted(missing_tests)}"
assert test_result.returncode == 0, "dbt test có test thất bại."
assert test_summary["status"].eq("PASS").all(), (
    "Có test Gold không PASS; xem log dbt ở trên."
)

11:41:06  Running with dbt=1.12.6
11:41:07  Registered adapter: duckdb=1.11.1
11:41:07  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 1 unused configuration paths:
- models.commerceflow360.gold.marts
11:41:07  Found 19 models, 75 data tests, 10 sources, 522 macros
11:41:07  
11:41:07  Concurrency: 4 threads (target='dev')
11:41:07  
11:41:07  1 of 3 START test test_gold_dimension_relationships ............................ [RUN]
11:41:07  2 of 3 START test test_gold_fact_grain ......................................... [RUN]
11:41:07  3 of 3 START test test_sales_reconciliation .................................... [RUN]
11:41:08  1 of 3 PASS test_gold_dimension_relationships .................................. [PASS in 0.11s]
11:41:08  2 of 3 PASS test_gold_fact_grain ............................................... [PASS in 0.11s]
11:41:08  3 of 3 PASS test_sales_reconciliation .......................................... [P

,test_name,status,failures
0,test_gold_dimension_relationships,PASS,0
1,test_gold_fact_grain,PASS,0
2,test_sales_reconciliation,PASS,0


Cell 3 — bảng đối soát để đưa vào báo cáo

In [3]:
reconciliation_sql = """
with reconciliations as (

    select
        'order_count' as metric,
        cast(
            (
                select count(distinct lower(trim(cast(order_id as varchar))))
                from bronze_orders
            ) as double
        ) as source_value,
        cast((select count(*) from silver_orders) as double) as silver_value,
        cast((select sum(order_count) from fact_orders) as double) as gold_value

    union all

    select
        'item_revenue',
        cast((select sum(try_cast(price as decimal(18, 2))) from bronze_order_items) as double),
        cast((select sum(price) from silver_order_items) as double),
        cast((select sum(item_price) from fact_order_items) as double)

    union all

    select
        'freight_value',
        cast((select sum(try_cast(freight_value as decimal(18, 2))) from bronze_order_items) as double),
        cast((select sum(freight_value) from silver_order_items) as double),
        cast((select sum(freight_value) from fact_order_items) as double)

    union all

    select
        'payment_value',
        cast((select sum(try_cast(payment_value as decimal(18, 2))) from bronze_order_payments) as double),
        cast((select sum(payment_value) from silver_order_payments) as double),
        cast((select sum(payment_value) from fact_payments) as double)
)
select
    metric,
    source_value,
    silver_value,
    gold_value,
    abs(source_value - silver_value) as source_to_silver_difference,
    abs(silver_value - gold_value) as silver_to_gold_difference
from reconciliations
order by metric
"""

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as connection:
    reconciliation = connection.execute(reconciliation_sql).fetchdf()

display(reconciliation)

assert not reconciliation[
    ["source_value", "silver_value", "gold_value"]
].isna().any().any(), "Có số liệu đối soát bị NULL."

assert reconciliation[
    ["source_to_silver_difference", "silver_to_gold_difference"]
].lt(0.01).all().all(), "Có chênh lệch vượt ngưỡng 0.01."

,metric,source_value,silver_value,gold_value,source_to_silver_difference,silver_to_gold_difference
0,freight_value,2251909.54,2251909.54,2251909.54,0.0,0.0
1,item_revenue,13591643.70,13591643.70,13591643.70,0.0,0.0
2,order_count,99441.00,99441.00,99441.00,0.0,0.0
3,payment_value,16008872.12,16008872.12,16008872.12,0.0,0.0


Cell 4 — xuất kết quả để dùng cho báo cáo

In [4]:
output_dir = PROJECT_ROOT / "reports" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "Bang_3.7.1_DoiSoatGold.csv"
reconciliation.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Đã xuất:", output_path)

Đã xuất: D:\KHOI\DAI HOC\TREN TRUONG\NAM3\KI 1\DATA WAREHOUSE\DOAN\CommerceFlow360\reports\tables\Bang_3.7.1_DoiSoatGold.csv
